In [ ]:
import os
import argparse
import torch
from datasets import MyDataSet
# from vit_model import VisionTransformer
from Residual import Residual
from Residual import Student


import collections
import math
import shutil
import pandas as pd
import numpy as np
import torchvision
from torch import nn
from torch.utils.data import Dataset
from torch.nn import functional as F
from d2l import torch as d2l
from PIL import Image


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
singlefile_data_num = 400  #每个文件训练读取数据个数
singlefile_val_data_num = 100 #每个文件验证读取数据个数
batch_size = 64
epochs = 101
lr = 0.005    #学习率大小
train_path = r"E:\北邮2024\本科毕设\Model\Data\srs_5points_20cycles\train"
test_path =  r"E:\北邮2024\本科毕设\Model\Data\srs_5points_20cycles\test"
if os.path.exists("./weights") is False:
    os.makedirs("./weights")

In [ ]:
# 实例化训练数据集
print("读取训练集数据，每个文件读{}条数据".format(singlefile_data_num))
train_dataset = MyDataSet(folder_path=train_path)

In [ ]:
# 实例化测试数据集
print("读取验证集数据，每个文件读{}条数据".format(singlefile_val_data_num))
val_dataset = MyDataSet(folder_path = test_path)

In [ ]:

nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])  # number of workers
nw=0 # in windows
print('Using {} dataloader workers every process'.format(nw))

train_loader = torch.utils.data.DataLoader(train_dataset,
                                            batch_size=batch_size,  #weight_decay=1e-3
                                            shuffle=True,
                                            pin_memory=True,
                                            num_workers=nw)

val_loader = torch.utils.data.DataLoader(val_dataset,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            pin_memory=True,
                                            num_workers=nw)

In [ ]:
# 清空txt数据
with open("loss.txt", "w") as f_loss:
    f_loss.write("")
with open("accuracy.txt", "w") as f_accuracy:
    f_accuracy.write("")
with open("accuracy_test.txt", "w") as f:
    f.write("")
with open("loss_test.txt", "w") as f:
    f.write("")

In [ ]:
print(epochs)
from utils import train_one_epoch,test_model
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
model=Student().to(device)
model.train()
optimizer = torch.optim.SGD(model.parameters(), lr = lr,weight_decay=1e-3)
for epoch in range(epochs):
    # train
    train_loss, train_accuracy = train_one_epoch(model=model,
                                optimizer=optimizer,
                                data_loader=train_loader,
                                device=device,
                                epoch=epoch)
    optimizer.step()
    with open("loss.txt", "a") as f_loss:
        f_loss.write("loss:{}\n".format(train_loss))
    with open("accuracy.txt", "a") as f_accuracy:
        f_accuracy.write("accuracy:{}\n".format(train_accuracy))
    # validate
    if (epoch+1) % 5 == 0:
        pred, test_loss, labels, test_accuracy = test_model(model=model,
                                data_loader=val_loader,
                                device=device)
        average_loss = sum(test_loss)/len(test_loss)
        print("................")
        print("验证集结果：")
        print(f"平均误差: {average_loss:.3f}")
        print(f"准确率: {test_accuracy:.2f}%")
        print("................")
        with open("accuracy_test.txt", "a") as f:
            f.write("accuracy test:{}\n".format(test_accuracy))
        with open("loss_test.txt", "a") as f:
            f.write("loss test:{}\n".format(average_loss))
        
    if (epoch+1) % 20 == 0:
        print("保存模型")
        # torch.save(model.state_dict(), "./weights/model-{}.pth".format(epoch))
        if not os.path.exists('./model_save'):
            os.makedirs('./model_save')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': train_loss,
            'accuracy': train_accuracy
            }, "./model_save/model-{}.pth".format(epoch))
print("训练完成")

101
 
[train epoch 1] 平均误差:1.580, 平均准确率:27.49%
................
验证集结果：
平均误差: 1.577
准确率: 32.29%
................
 
[train epoch 2] 平均误差:1.539, 平均准确率:30.20%
 
[train epoch 3] 平均误差:1.522, 平均准确率:31.57%
 
[train epoch 4] 平均误差:1.511, 平均准确率:32.71%
 
[train epoch 5] 平均误差:1.504, 平均准确率:33.41%
 
[train epoch 6] 平均误差:1.498, 平均准确率:33.90%
................
验证集结果：
平均误差: 1.524
准确率: 44.97%
................
 
[train epoch 7] 平均误差:1.496, 平均准确率:35.33%
 
[train epoch 8] 平均误差:1.491, 平均准确率:34.84%
 
[train epoch 9] 平均误差:1.483, 平均准确率:35.61%
 
[train epoch 10] 平均误差:1.478, 平均准确率:35.49%
 
[train epoch 11] 平均误差:1.473, 平均准确率:36.36%
................
验证集结果：
平均误差: 1.502
准确率: 41.15%
................
 
[train epoch 12] 平均误差:1.469, 平均准确率:37.56%
 
[train epoch 13] 平均误差:1.468, 平均准确率:37.20%
 
[train epoch 14] 平均误差:1.457, 平均准确率:38.47%
 
[train epoch 15] 平均误差:1.451, 平均准确率:37.89%
 
[train epoch 16] 平均误差:1.442, 平均准确率:39.33%
................
验证集结果：
平均误差: 1.484
准确率: 33.33%
................
 
[train epoch 17] 平均误差:1.436, 平均准确率:39.7

KeyboardInterrupt: 

In [ ]:
# from torchinfo import summary

# model = Student()
# summary(model, input_size=(1, 1, 256, 8), col_names=["input_size", "output_size", "num_params", "kernel_size"])